# Taller: Datos abiertos de Colombia 🇨🇴

Explora **datos.gov.co** con un *skill* de Claude — todo en el navegador, sin instalar nada en tu máquina.

**Cómo usar:** menú *Entorno de ejecución → Ejecutar todo*, o corre cada celda con **Shift+Enter**.

## 1. Traer el proyecto

In [ ]:
!git clone --depth 1 https://github.com/recoveredfactory/dagster-colombia-skill.git 2>/dev/null || echo 'ya estaba clonado'
%cd dagster-colombia-skill

## 2. El skill en acción (solo Python, sin instalar nada)

El skill usa solo la librería estándar. Buscamos un dataset y luego pedimos un ranking por departamento.

In [ ]:
CLI = '.claude/skills/colombia-open-data/scripts/cli.py'
!python3 {CLI} search "internet por departamento" --limit 3

In [ ]:
!python3 {CLI} query n48w-gutb --select "departamento,sum(no_de_accesos::number) as accesos" --where "anno='2023' and trimestre='3'" --group departamento --order "accesos desc" --limit 5

## 3. Procesar y visualizar con pandas (ya viene en Colab)

Importamos el módulo del skill directamente y graficamos el ranking.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '.claude/skills/colombia-open-data/scripts')
import socrata

rows = socrata.query(
    'n48w-gutb',
    select='departamento,sum(no_de_accesos::number) as accesos',
    where="anno='2023' and trimestre='3'",
    group='departamento', order='accesos desc',
)
df = pd.DataFrame(rows)
df['accesos'] = pd.to_numeric(df['accesos'])
ax = df.head(10).plot.barh(x='departamento', y='accesos', legend=False,
                           title='Suscriptores de internet fijo por departamento (2023-T3)')
ax.invert_yaxis()

## 4. ¿Y el pipeline completo?

La orquestación con **Dagster** y la web en **Next.js** se corren mejor localmente — sigue el `README.md`. Aquí ya viste lo esencial: **adquirir → procesar → visualizar**.

Para instalar las dependencias del pipeline (si quieres experimentar aquí):

In [ ]:
!pip install -q -r pipeline/requirements.txt
print('listo')